# Introduction to QSAR Modeling
### A Beginner-Friendly Walkthrough in Google Colab

**QSAR** (Quantitative Structure–Activity Relationship) modeling is a computational technique used in drug discovery and cheminformatics to relate the **chemical structure** of a molecule to its **biological activity** (e.g. IC50, potency, toxicity).

The core idea:

> *Molecules with similar structures tend to have similar biological activities.*

**Typical QSAR workflow:**
1. Collect a dataset of molecules (SMILES) with known biological activity
2. Calculate **molecular descriptors** / **fingerprints** that numerically represent chemical structure (RDKit, PaDEL)
3. Explore and clean the data (pandas, seaborn, matplotlib)
4. Build machine learning models that predict activity from descriptors
5. Evaluate and interpret the model

This notebook walks through each step using a small example dataset so you can run everything end-to-end in class.

---


## 1. Meet the Packages

Before installing anything, let's understand **what each package does and why it matters for QSAR**. Think of a QSAR pipeline as an assembly line — each package handles one station.

| Package | Category | Role in QSAR |
|---|---|---|
| **pandas** | Data handling | Stores and manipulates our dataset as tables (DataFrames) — molecules, descriptors, activity values, all in rows/columns |
| **numpy** | Data handling | Underlying numerical arrays and math operations that other libraries rely on |
| **seaborn** | Visualization | Statistical plots (histograms, scatter plots, heatmaps) with attractive defaults — great for EDA |
| **matplotlib** | Visualization | The core plotting engine that seaborn (and everything else) is built on; gives fine-grained control over figures |
| **RDKit** | Cheminformatics | The most widely-used open-source cheminformatics toolkit. Reads SMILES, draws molecules, and calculates **molecular descriptors** (e.g. molecular weight, LogP) directly in Python |
| **PaDEL-Descriptor** (via `padelpy`) | Cheminformatics | A Java-based tool (used through Python) that calculates additional **2D descriptors and fingerprints** — often used alongside or instead of RDKit's own descriptors, especially in published QSAR papers |
| **scikit-learn** | Machine learning | Provides the train/test splitting tools, the actual ML algorithms (e.g. Random Forest), and evaluation metrics (R², RMSE) |
| **LazyPredict** | Machine learning | An "AutoML" convenience package — trains and compares **20-30 regression/classification models at once** with one function call, so you can quickly see which algorithm family looks promising before tuning one by hand |

**Why do we need both RDKit *and* PaDEL?**
They both turn a molecule's structure into numbers, but they were developed independently and don't compute the exact same set of descriptors. RDKit is pure Python and very fast; PaDEL (originally a standalone Java GUI tool) is what many published QSAR studies used historically, so knowing both is useful when reproducing or comparing against literature models.

Now let's install them.

## 2. Install Required Packages

Run this cell first. It installs everything described above:

- **RDKit** — compute molecular descriptors & fingerprints from SMILES
- **PaDELPy** — Python wrapper for PaDEL-Descriptor (another descriptor/fingerprint calculator popular in QSAR)
- **LazyPredict** — quickly benchmark dozens of ML models with a couple of lines of code
- **pandas, seaborn, matplotlib, scikit-learn** — data handling, visualization, and modeling

> This may take 1-3 minutes on first run. Restart the runtime if prompted after installation.

In [ ]:
# Core scientific / ML packages
!pip install -q pandas seaborn matplotlib scikit-learn

# Cheminformatics packages
!pip install -q rdkit
!pip install -q padelpy

# AutoML-style model comparison
!pip install -q lazypredict


In [ ]:
# (Optional) Check that Java is available -- PaDEL-Descriptor needs a Java runtime
!java -version


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, Lipinski
from rdkit.Chem.Draw import IPythonConsole

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline


## 4. The Dataset

For this introductory lecture we use a small, self-contained example dataset of molecules (given as **SMILES strings**) along with a bioactivity value expressed as **pIC50** (`-log10(IC50 in M)` — higher pIC50 = more potent).

> In a real project you would typically pull a larger dataset from a public database such as **ChEMBL** (e.g. via the `chembl_webresource_client` package). A short example of how to do that is included at the very end of this notebook (Section 9) for reference.

In [ ]:
# A small illustrative dataset (SMILES + pIC50)
data = {
    "compound_id": [f"CMP{i+1:03d}" for i in range(15)],
    "smiles": [
        "CC(=O)Oc1ccccc1C(=O)O",              # Aspirin
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",         # Ibuprofen
        "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",       # Caffeine
        "CC(=O)Nc1ccc(O)cc1",                 # Paracetamol
        "c1ccc2c(c1)ccc(=O)o2",               # Coumarin
        "CC1=CC(=O)C=CC1=O",                  # Example quinone
        "Clc1ccccc1Nc1ccccc1C(=O)O",          # Diclofenac
        "COc1ccc2cc(ccc2c1)C(C)C(=O)O",       # Naproxen
        "O=C(Nc1ccccc1)Nc1ccccc1",            # Carbanilide
        "CCOC(=O)c1ccccc1",                   # Ethyl benzoate
        "CCN(CC)C(=O)c1ccccc1",               # Example amide
        "c1ccc(cc1)C(=O)Nc1ccccc1",           # Benzanilide
        "COc1cc2c(cc1OC)C(=O)C(=CO2)c1ccccc1",# Example flavonoid
        "CC(C)NCC(O)c1ccc(O)c(O)c1",          # Isoproterenol-like
        "CCC(C)C1(CC)C(=O)NC(=O)NC1=O"        # Barbiturate-like
    ],
    "pIC50": [5.2, 5.8, 4.9, 5.5, 6.1, 4.7, 6.4, 5.9, 5.0, 4.5,
              5.3, 5.1, 6.0, 5.7, 4.8]
}

df = pd.DataFrame(data)
df.head()


## 5. Sanity-Check the Molecules with RDKit

Before computing descriptors, always confirm that every SMILES string parses correctly.

In [ ]:
def is_valid_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

df["valid_smiles"] = df["smiles"].apply(is_valid_smiles)
print(df["valid_smiles"].value_counts())

# Drop any invalid entries (none expected here, but good practice)
df = df[df["valid_smiles"]].reset_index(drop=True)


In [ ]:
# Visualize the first few molecules
mols = [Chem.MolFromSmiles(s) for s in df["smiles"][:6]]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(220, 220),
                      legends=df["compound_id"][:6].tolist())


## 6. Molecular Descriptors with RDKit

**Descriptors** are numerical properties calculated from a molecule's structure. A classic, easy-to-interpret set is used in **Lipinski's Rule of Five**, which is often introduced early in QSAR/drug-discovery courses:

- **Molecular Weight (MW)**
- **LogP** (lipophilicity)
- **Number of Hydrogen Bond Donors (NumHDonors)**
- **Number of Hydrogen Bond Acceptors (NumHAcceptors)**

In [ ]:
def lipinski_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    h_donors = Lipinski.NumHDonors(mol)
    h_acceptors = Lipinski.NumHAcceptors(mol)
    return pd.Series([mw, logp, h_donors, h_acceptors])

df[["MW", "LogP", "NumHDonors", "NumHAcceptors"]] = df["smiles"].apply(lipinski_descriptors)
df.head()


In [ ]:
# A wider set of RDKit descriptors (200+) can be computed automatically
from rdkit.ML.Descriptors import MoleculeDescriptors

descriptor_names = [d[0] for d in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

def compute_all_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return calculator.CalcDescriptors(mol)

rdkit_desc = df["smiles"].apply(compute_all_rdkit_descriptors)
rdkit_desc_df = pd.DataFrame(rdkit_desc.tolist(), columns=descriptor_names)
print(f"Computed {rdkit_desc_df.shape[1]} RDKit descriptors for {rdkit_desc_df.shape[0]} molecules")
rdkit_desc_df.iloc[:, :8].head()


## 7. Descriptors / Fingerprints with PaDEL (via `padelpy`)

**PaDEL-Descriptor** is a widely-used Java-based tool (popularized by many QSAR tutorials) that calculates **1D/2D descriptors** and several types of **molecular fingerprints** (e.g. PubChem fingerprints, substructure fingerprints). `padelpy` gives us a simple Python interface to it.

Below we:
1. Save our molecules as a `.smi` file (PaDEL's expected input format)
2. Call PaDEL to compute descriptors
3. Load the results back into a DataFrame

In [ ]:
from padelpy import padeldescriptor

# PaDEL can also compute fingerprints (e.g. PubChem fingerprints) instead of / alongside descriptors
padeldescriptor(
    mol_dir="molecules.smi",
    d_file="padel_fingerprints.csv",
    d_2d=False,
    fingerprints=True,
    retainorder=True
)

padel_fp_df = pd.read_csv("padel_fingerprints.csv")
print(padel_fp_df.shape)
padel_fp_df.iloc[:, :8].head()

## 8. Exploratory Data Analysis (EDA)

Before modeling, it's good practice to visualize the relationship between descriptors and the target activity (`pIC50`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["pIC50"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of pIC50")

sns.scatterplot(data=df, x="MW", y="pIC50", ax=axes[1], color="darkorange")
axes[1].set_title("MW vs pIC50")

sns.scatterplot(data=df, x="LogP", y="pIC50", ax=axes[2], color="seagreen")
axes[2].set_title("LogP vs pIC50")

plt.tight_layout()
plt.show()


## 9. Build the QSAR Model

We'll combine the descriptors into a feature matrix `X` and use `pIC50` as the target `y`, then:

1. Split the data into train/test sets
2. Use **LazyPredict** to quickly compare many regression algorithms
3. Train a chosen model (Random Forest) and evaluate it

In [ ]:
# Combine RDKit Lipinski descriptors with the wider RDKit descriptor set
X = pd.concat([df[["MW", "LogP", "NumHDonors", "NumHAcceptors"]], rdkit_desc_df], axis=1)
X = X.loc[:, ~X.columns.duplicated()]          # remove any duplicate columns
X = X.fillna(0)                                 # simple handling of missing values
y = df["pIC50"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


In [ ]:
# --- LazyPredict: benchmark many regressors in a few lines of code ---
from lazypredict.Supervised import LazyRegressor

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

models.sort_values("R-Squared", ascending=False).head(10)


In [ ]:
# --- Train a Random Forest model explicitly (a common baseline in QSAR) ---
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R^2  : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")

In [ ]:
# Visualize predicted vs experimental pIC50
plt.figure(figsize=(5, 5))
sns.scatterplot(x=y_test, y=y_pred, s=80, color="purple")
lims = [min(y_test.min(), y_pred.min()) - 0.2, max(y_test.max(), y_pred.max()) + 0.2]
plt.plot(lims, lims, "k--", linewidth=1)
plt.xlabel("Experimental pIC50")
plt.ylabel("Predicted pIC50")
plt.title("Random Forest: Predicted vs Experimental")
plt.xlim(lims); plt.ylim(lims)
plt.tight_layout()
plt.show()
